# B2-023-generative-models-diffusion — Practice p16 — Solution

**Type:** proof · **Difficulty:** advanced · **Concepts:** denoising-diffusion-probabilistic-models

*45 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

## Solution

### (a) Equal-variance KL

Put $\sigma_j=\tau_j=\sigma>0$ for every coordinate $j=1,\dots,d$ in the diagonal closed form:
$$\sum_j\Big[\log\frac\sigma\sigma+\frac{\sigma^2+(\mu_{1j}-\mu_{2j})^2}{2\sigma^2}-\frac12\Big]=\sum_j\Big[0+\frac12+\frac{(\mu_{1j}-\mu_{2j})^2}{2\sigma^2}-\frac12\Big]=\frac{\lVert\mu_1-\mu_2\rVert^2}{2\sigma^2}.$$

### (b1) The posterior mean in terms of $\varepsilon$ ($t\ge2$)

Write $c_0=\frac{\sqrt{\bar\alpha_{t-1}}\,\beta_t}{1-\bar\alpha_t}$, $c_1=\frac{\sqrt{\alpha_t}(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}$ and substitute $x_0=(x_t-\sqrt{1-\bar\alpha_t}\,\varepsilon)/\sqrt{\bar\alpha_t}$.
Because $\bar\alpha_t=\alpha_t\bar\alpha_{t-1}$, $\ \frac{\sqrt{\bar\alpha_{t-1}}}{\sqrt{\bar\alpha_t}}=\frac1{\sqrt{\alpha_t}}$, so $c_0/\sqrt{\bar\alpha_t}=\frac{\beta_t}{\sqrt{\alpha_t}(1-\bar\alpha_t)}$ and
$$\tilde\mu_t=\Big[\frac{\beta_t}{\sqrt{\alpha_t}(1-\bar\alpha_t)}+\frac{\sqrt{\alpha_t}(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}\Big]x_t-\frac{\beta_t\sqrt{1-\bar\alpha_t}}{\sqrt{\alpha_t}(1-\bar\alpha_t)}\,\varepsilon .$$
The $x_t$ coefficient is $\frac{\beta_t+\alpha_t(1-\bar\alpha_{t-1})}{\sqrt{\alpha_t}(1-\bar\alpha_t)}$, and the **coefficient identity**
$$\beta_t+\alpha_t(1-\bar\alpha_{t-1})=(1-\alpha_t)+\alpha_t-\alpha_t\bar\alpha_{t-1}=1-\bar\alpha_t$$
(which uses only $\beta_t=1-\alpha_t$ and $\alpha_t\bar\alpha_{t-1}=\bar\alpha_t$) reduces it to $1/\sqrt{\alpha_t}$. The $\varepsilon$ coefficient simplifies to $\frac{\beta_t}{\sqrt{\alpha_t}\sqrt{1-\bar\alpha_t}}$. Hence
$$\tilde\mu_t=\frac1{\sqrt{\alpha_t}}\Big(x_t-\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\varepsilon\Big).$$

### (b2) The KL as a weighted ε error

For $t\ge2$, $\tilde\beta_t>0$ (because $\bar\alpha_{t-1}<1$), and both $q(x_{t-1}\mid x_t,x_0)=N(\tilde\mu_t,\tilde\beta_tI)$ and $p_\theta(x_{t-1}\mid x_t)=N(\mu_\theta,\tilde\beta_tI)$ have the same variance. Subtracting the two mean formulas, the $x_t/\sqrt{\alpha_t}$ terms cancel:
$$\tilde\mu_t-\mu_\theta=\frac{\beta_t}{\sqrt{\alpha_t}\sqrt{1-\bar\alpha_t}}\big(\varepsilon_\theta(x_t,t)-\varepsilon\big).$$
By (a) with $\sigma^2=\tilde\beta_t$,
$$\mathrm{KL}=\frac{\lVert\tilde\mu_t-\mu_\theta\rVert^2}{2\tilde\beta_t}=\lambda_t\lVert\varepsilon-\varepsilon_\theta(x_t,t)\rVert^2,\qquad\lambda_t=\frac{\beta_t^2}{2\,\tilde\beta_t\,\alpha_t\,(1-\bar\alpha_t)} .$$

### (c) Why $t=1$ is excluded

Under this unit's convention $\bar\alpha_0=1$, so $\tilde\beta_1=\frac{1-\bar\alpha_0}{1-\bar\alpha_1}\beta_1=0$: the posterior $q(x_0\mid x_1,x_0)$ is a point mass at $x_0$, and the model's reverse step with $\sigma_1^2=\tilde\beta_1=0$ is degenerate too. The formula of (a) needs $\sigma>0$ (it divides by $2\sigma^2$), so the argument simply does not apply; there is no "$t=1$ KL term" of this form.
In the variational bound, the $t=1$ term is instead the **decoder likelihood** (reconstruction) term $-E_q[\log p_\theta(x_0\mid x_1)]$, which needs its own nonzero variance and is not computed in this unit.
The simplified objective still includes $\lVert\varepsilon-\varepsilon_\theta(x_1,1)\rVert^2$ at $t=1$, but that term is a **surrogate**: it is an unweighted squared error chosen because it trains the network well, not the log-density of $x_0$ under any stated Gaussian (it has no variance, no normalizing constant, and no weight derived from the bound), so its value is not a likelihood.

### (d) Sampling variance $\sigma_t^2=\beta_t$

Now the two Gaussians have variances $\tilde\beta_t$ (posterior) and $\beta_t$ (model). The diagonal closed form with $\sigma_j^2=\tilde\beta_t$, $\tau_j^2=\beta_t$, summed over $d$ coordinates, gives
$$\mathrm{KL}=\underbrace{\frac d2\Big[\log\frac{\beta_t}{\tilde\beta_t}+\frac{\tilde\beta_t}{\beta_t}-1\Big]}_{\text{(i) independent of the model}}+\frac{\lVert\tilde\mu_t-\mu_\theta\rVert^2}{2\beta_t}.$$
(ii) The coefficient of the squared mean error becomes $\frac1{2\beta_t}$ instead of $\frac1{2\tilde\beta_t}$; in terms of $\varepsilon$ the weight is $\frac{\beta_t^2}{2\beta_t\alpha_t(1-\bar\alpha_t)}=\frac{\beta_t}{2\alpha_t(1-\bar\alpha_t)}$.
Yes: up to the model-independent constant (which has zero gradient), the loss is still an ε squared error, with a different per-$t$ weight.

### (e) Numbers: $T=4$, linear from $0.05$ to $0.35$

$\beta=(0.05,0.15,0.25,0.35)$, $\bar\alpha_2=0.8075$, $\bar\alpha_3=0.605625$, $\alpha_3=0.75$.
$$\tilde\beta_3=\frac{1-0.8075}{1-0.605625}\cdot0.25=\frac{0.1925}{0.394375}\cdot0.25=0.122029,\qquad\lambda_3=\frac{0.25^2}{2\cdot0.122029\cdot0.75\cdot0.394375}=0.865801 .$$
(Exactly, $\lambda_3=\frac{\beta_3}{2\alpha_3(1-\bar\alpha_{2})}=\frac{0.25}{2\cdot0.75\cdot0.1925}$, since $\tilde\beta_3(1-\bar\alpha_3)=(1-\bar\alpha_2)\beta_3$.)

In [ ]:
import math

import torch

T = 4
beta = torch.zeros(T + 1, dtype=torch.float64)
beta[1:] = torch.linspace(0.05, 0.35, T, dtype=torch.float64)
alpha = 1 - beta
alpha_bar = torch.cumprod(alpha, dim=0)
beta_tilde = torch.zeros(T + 1, dtype=torch.float64)
beta_tilde[1:] = (1 - alpha_bar[:-1]) / (1 - alpha_bar[1:]) * beta[1:]
t = 3
lam = (beta[t] ** 2 / (2 * beta_tilde[t] * alpha[t] * (1 - alpha_bar[t]))).item()

# numeric check of (b1) and (b2) on random vectors
g = torch.Generator().manual_seed(20261015)
x0 = torch.randn(2, generator=g, dtype=torch.float64)
eps = torch.randn(2, generator=g, dtype=torch.float64)
eps_theta = torch.randn(2, generator=g, dtype=torch.float64)
x_t = alpha_bar[t].sqrt() * x0 + (1 - alpha_bar[t]).sqrt() * eps
mu_tilde_def = (alpha_bar[t - 1].sqrt() * beta[t] / (1 - alpha_bar[t]) * x0
                + alpha[t].sqrt() * (1 - alpha_bar[t - 1]) / (1 - alpha_bar[t]) * x_t)
mu_tilde_eps = (x_t - beta[t] / (1 - alpha_bar[t]).sqrt() * eps) / alpha[t].sqrt()
mu_theta = (x_t - beta[t] / (1 - alpha_bar[t]).sqrt() * eps_theta) / alpha[t].sqrt()


def kl_diag(mu, var_p, nu, var_q):
    return (0.5 * torch.log(var_q / var_p) + (var_p + (mu - nu) ** 2) / (2 * var_q) - 0.5).sum()


kl_equal = kl_diag(mu_tilde_def, beta_tilde[t], mu_theta, beta_tilde[t]).item()
kl_lambda = lam * ((eps - eps_theta) ** 2).sum().item()
kl_beta = kl_diag(mu_tilde_def, beta_tilde[t], mu_theta, beta[t]).item()
const_beta = (2 / 2 * (torch.log(beta[t] / beta_tilde[t]) + beta_tilde[t] / beta[t] - 1)).item()
kl_beta_formula = const_beta + (beta[t] / (2 * alpha[t] * (1 - alpha_bar[t])) * ((eps - eps_theta) ** 2).sum()).item()
print(f"beta_tilde_3 {beta_tilde[t].item():.6f}  lambda_3 {lam:.6f}")
print(f"KL equal-var {kl_equal:.10f} vs lambda*||.||^2 {kl_lambda:.10f};  KL sigma^2=beta {kl_beta:.10f} vs {kl_beta_formula:.10f}")

### Answer check

In [ ]:
assert round(beta_tilde[t].item(), 6) == 0.122029
assert round(lam, 6) == 0.865801
assert math.isclose(lam, 0.25 / (2 * 0.75 * 0.1925), rel_tol=1e-12, abs_tol=0.0)
assert torch.allclose(mu_tilde_def, mu_tilde_eps, atol=1e-12, rtol=0)
assert math.isclose(kl_equal, kl_lambda, rel_tol=1e-10, abs_tol=0.0)
assert math.isclose(kl_beta, kl_beta_formula, rel_tol=1e-10, abs_tol=0.0)
assert beta_tilde[1].item() == 0.0